# RFW Face-Verification Fairness Baseline

This notebook computes face-verification accuracy disparity across 4 races
(African, Asian, Caucasian, Indian) using two embedding models:
- **ArcFace** via InsightFace `buffalo_m`
- **AdaFace** via the R50/WebFace4M checkpoint

**Dataset:** `chidroopakanaparthy/rfw-clean-400x400` (must be added to this notebook as an input).

**Hardware:** Requires GPU runtime.

In [ ]:
# =============================================================================
# CELL 1 — ENVIRONMENT SETUP (Step 1)
# =============================================================================
# This cell:
#   1. Confirms GPU availability via nvidia-smi
#   2. Installs all required packages (with specific ONNX version pinning)
#   3. Verifies imports
#
# Expected output: nvidia-smi GPU info, pip install logs, and
#   "All imports successful. Environment ready." at the end.
# =============================================================================

import subprocess, sys

# 1a. Confirm GPU
print("=" * 60)
print("GPU CHECK")
print("=" * 60)
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("nvidia-smi failed — no GPU detected. "
                       "Please switch to a GPU runtime.")

# 1b. Install packages
print("\n" + "=" * 60)
print("INSTALLING PACKAGES")
print("=" * 60)

# Clean up conflicting onnxruntime installs first to ensure CUDA provider binds correctly
print("Cleaning old onnxruntime installations...")
subprocess.run(
    "pip uninstall -y onnxruntime onnxruntime-gpu onnxruntime-directml onnxruntime-azure 2>/dev/null", 
    shell=True
)

# --break-system-packages handles PEP 668 externally-managed envs on Kaggle
packages = [
    "insightface",
    "onnxruntime-gpu==1.20.1", # Pinned for CUDA 12.8 compatibility on Kaggle
    "opencv-python-headless",
    "scikit-learn",
    "pandas",
    "matplotlib",
    "scipy",
    "gdown",           # for downloading AdaFace checkpoint from Google Drive
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--break-system-packages",
     "--quiet"] + packages
)

# 1c. Quick import smoke test
import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import scipy
import insightface

print(f"\nPyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device    : {torch.cuda.get_device_name(0)}")
print(f"OpenCV         : {cv2.__version__}")
print(f"InsightFace    : {insightface.__version__}")
print(f"scikit-learn   : {sklearn.__version__}")
print(f"\n

In [ ]:
# =============================================================================
# CELL 2 — LOAD ARCFACE / buffalo_m (Step 2)
# =============================================================================
# This cell:
#   1. Initializes InsightFace's FaceAnalysis with the buffalo_m model pack
#   2. Auto-downloads buffalo_m on first run (~200 MB)
#   3. Fixes a known InsightFace download bug (nested folders)
#   4. Prints the embedding dimension (should be 512)
#
# Expected output:
#   Download progress (first run only), model loading logs,
#   "ArcFace (buffalo_m) loaded. Embedding dim: 512"
# =============================================================================

import os
import shutil
from insightface.app import FaceAnalysis
import cv2
import numpy as np

# Initialize with buffalo_m (contains detection + recognition models)
# ctx_id=0 → use first GPU if available. 
# We include CPU as a fallback in case Kaggle's CUDA version mismatches onnxruntime-gpu
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

try:
    arcface_app = FaceAnalysis(name='buffalo_m', providers=providers)
except AssertionError as e:
    # InsightFace often extracts buffalo_m.zip into a nested buffalo_m/buffalo_m folder,
    # causing a silent failure and an AssertionError: assert 'detection' in self.models
    model_dir = os.path.expanduser("~/.insightface/models/buffalo_m")
    nested_dir = os.path.join(model_dir, "buffalo_m")
    if os.path.isdir(nested_dir):
        print("⚠️ InsightFace nested directory bug detected! Fixing...")
        for filename in os.listdir(nested_dir):
            shutil.move(os.path.join(nested_dir, filename), os.path.join(model_dir, filename))
        shutil.rmtree(nested_dir)
        print("✅ Fixed corrupted model directory structure. Retrying load...")
        arcface_app = FaceAnalysis(name='buffalo_m', providers=providers)
    else:
        raise e

arcface_app.prepare(ctx_id=0, det_size=(640, 640))

# Verify it works: create a dummy image and check embedding shape
# We'll just confirm the recognition model is present
rec_model = None
for model in arcface_app.models.values():
    if hasattr(model, 'embedding_dim'):
        rec_model = model
        break

if rec_model is not None:
    arcface_emb_dim = rec_model.embedding_dim
else:
    # Fallback: insightface buffalo_m always produces 512-d embeddings
    arcface_emb_dim = 512

print(f"\n✅ ArcFace (buffalo_m) loaded successfully.")
print(f"   Embedding dimension: {arcface_emb_dim}")
print(f"   Detection model(s):  {list(arcface_app.models.keys())}")


In [ ]:
# =============================================================================
# CELL 3 — LOAD ADAFACE / WebFace4M R50 (Step 3)
# =============================================================================
# This cell:
#   1. Clones the AdaFace repo
#   2. Downloads the R50/WebFace4M checkpoint from Google Drive via gdown
#   3. Loads the model using AdaFace's own code
#   4. Sets up MTCNN for AdaFace-compatible face detection + alignment
#
# Expected output:
#   git clone output, gdown download progress, model load confirmation,
#   "AdaFace (R50 WebFace4M) loaded. Embedding dim: 512"
#
# ⚠️  If gdown fails with a "Too many requests" or download quota error,
#    STOP — you'll need to manually download the checkpoint and upload it
#    as a Kaggle dataset instead.
# =============================================================================

import os
import subprocess
import sys
import torch

# ---------- 3a. Clone AdaFace repo ----------
ADAFACE_DIR = "/kaggle/working/AdaFace"
if not os.path.exists(ADAFACE_DIR):
    print("Cloning AdaFace repository...")
    subprocess.check_call(
        ["git", "clone", "https://github.com/mk-minchul/AdaFace.git", ADAFACE_DIR]
    )
    print("✅ AdaFace repo cloned.")
else:
    print("AdaFace repo already exists, skipping clone.")

# ---------- 3b. Download checkpoint ----------
CKPT_DIR = os.path.join(ADAFACE_DIR, "pretrained")
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "adaface_ir50_webface4m.ckpt")

# Google Drive file ID for AdaFace R50 WebFace4M
GDRIVE_FILE_ID = "1BmDRrhPsHSbXcWZoYFPJg2KJn1sd3QpN"

if not os.path.exists(CKPT_PATH):
    print("\nDownloading AdaFace R50/WebFace4M checkpoint from Google Drive...")
    try:
        import gdown
        gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}",
            CKPT_PATH,
            quiet=False
        )
        if not os.path.exists(CKPT_PATH) or os.path.getsize(CKPT_PATH) < 1_000_000:
            raise RuntimeError(
                "Download appears to have failed (file too small or missing).\n"
                "This usually means Google Drive blocked the download due to \n"
                "quota limits or Kaggle network restrictions.\n\n"
                "👉 MANUAL FIX: Download the checkpoint locally from:\n"
                f"   https://drive.google.com/file/d/{GDRIVE_FILE_ID}/view\n"
                "   Then upload it as a small Kaggle dataset and update CKPT_PATH below."
            )
        print(f"✅ Checkpoint downloaded: {CKPT_PATH}")
        print(f"   Size: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB")
    except Exception as e:
        print(f"\n❌ gdown download FAILED: {e}")
        print("\n" + "=" * 60)
        print("MANUAL WORKAROUND REQUIRED")
        print("=" * 60)
        print(f"1. Download checkpoint from:")
        print(f"   https://drive.google.com/file/d/{GDRIVE_FILE_ID}/view")
        print(f"2. Upload it as a Kaggle dataset")
        print(f"3. Update CKPT_PATH to point to the uploaded file")
        print(f"4. Re-run this cell")
        raise
else:
    print(f"Checkpoint already exists: {CKPT_PATH}")

# ---------- 3c. Load AdaFace model ----------
# Add AdaFace repo to path so we can import its modules
if ADAFACE_DIR not in sys.path:
    sys.path.insert(0, ADAFACE_DIR)

# Import AdaFace's network builder
import net as adaface_net

def load_adaface_model(architecture='ir_50', checkpoint_path=CKPT_PATH):
    """Load AdaFace model from checkpoint."""
    model = adaface_net.build_model(architecture)
    statedict = torch.load(checkpoint_path, map_location='cpu')['state_dict']
    # The checkpoint keys are prefixed with 'model.' — strip that prefix
    model_statedict = {
        key[6:]: val for key, val in statedict.items()
        if key.startswith('model.')
    }
    model.load_state_dict(model_statedict)
    model.eval()
    return model

adaface_model = load_adaface_model()
adaface_model = adaface_model.cuda()

print(f"\n✅ AdaFace (R50 WebFace4M) loaded on GPU.")
print(f"   Embedding dimension: 512")

# ---------- 3d. Set up MTCNN for AdaFace-compatible alignment ----------
# AdaFace uses MTCNN from its own face_alignment module for 112x112 crops.
# We import it from the cloned repo.
from face_alignment import align as adaface_align
from face_alignment import mtcnn as adaface_mtcnn

# Initialize MTCNN detector (112x112 aligned output)
mtcnn_model = adaface_mtcnn.MTCNN(device='cuda:0', crop_size=(112, 112))
print("✅ MTCNN face detector initialized for AdaFace alignment.")


def adaface_to_input(pil_rgb_image):
    """Convert a 112x112 aligned PIL RGB image to AdaFace BGR tensor.
    
    AdaFace expects:
      - BGR channel order (like OpenCV, not PIL's RGB)
      - Normalized to [-1, 1]
      - Shape: (1, 3, 112, 112)
    """
    import numpy as np
    np_img = np.array(pil_rgb_image)
    # RGB → BGR, normalize to [-1, 1]
    bgr_img = ((np_img[:, :, ::-1] / 255.0) - 0.5) / 0.5
    tensor = torch.tensor([bgr_img.transpose(2, 0, 1)]).float()
    return tensor

print("\n✅ All AdaFace components ready.")

In [ ]:
# ======================================================================
# CELL 4A — DIAGNOSTIC: Inspecting actual file naming conventions
# ======================================================================
import os
import glob

BASE_DIR = "/kaggle/input/datasets/chidroopakanaparthy/rfw-clean-400x400"
DATA_DIR = os.path.join(BASE_DIR, "data")
races = ["African", "Asian", "Caucasian", "Indian"]

for race in races:
    print(f"\n" + "─"*70)
    print(f"  RACE: {race}")
    print("─"*70)
    
    # 1. Find the pairs.txt file flexibly
    # It might be in txts/{race}/{race}_pairs.txt or just txts/{race}_pairs.txt
    search_pattern = os.path.join(BASE_DIR, "**", f"{race}_pairs.txt")
    found_txts = glob.glob(search_pattern, recursive=True)
    
    if not found_txts:
        print(f"  ❌ ERROR: Could not find {race}_pairs.txt anywhere in {BASE_DIR}")
        continue
        
    pairs_path = found_txts[0]
    print(f"  📄 Found pairs file at: {pairs_path}")
    print(f"  📄 First 5 lines of {os.path.basename(pairs_path)}:")
    print("-" * 73)
    with open(pairs_path, "r") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"      {repr(line)}")
    
    # 2. Inspect the data directory structure for this race
    race_dir = os.path.join(DATA_DIR, race)
    if not os.path.isdir(race_dir):
        print(f"\n  ❌ ERROR: Data directory missing: {race_dir}")
        continue
        
    identities = sorted(os.listdir(race_dir))
    if not identities:
        print(f"\n  ❌ ERROR: No identity folders found in {race_dir}")
        continue
        
    # Sample two identities (first and middle)
    sample_ids = [identities[0], identities[len(identities)//2]]
    
    print("\n  📸 Sample image filenames:")
    print("-" * 73)
    for ident in sample_ids:
        ident_dir = os.path.join(race_dir, ident)
        if not os.path.isdir(ident_dir):
            continue
        files = sorted(os.listdir(ident_dir))
        print(f"      Identity '{ident}' contains {len(files)} files.")
        if files:
            print(f"      First 3 files : {files[:3]}")
            print(f"      Last file     : {files[-1]}")

print("\n" + "="*70)
print("🛑 STOP! Copy the output of this cell and share it with me.")
print("This tells us the EXACT zero-padding and pairs format so we can")
print("hardcode Cell 4B correctly without guessing!")
print("="*70)


In [ ]:
# =============================================================================
# CELL 4B — PARSE PAIRS AND SAMPLE (Step 4)
# =============================================================================
# This cell:
#   1. Locates the RFW dataset (Kaggle input path)
#   2. Uses a robust find_image_path with STRICT matching:
#      - Primary: try {identity}_{img_id:04d}.jpg  (4-digit zero-pad)
#      - Fallback: list all files in identity folder, parse the numeric ID
#        from each filename, and match ONLY if int(parsed_id) == int(img_id)
#      - No substring glob matching — if no exact match, returns None
#   3. Tracks resolution method: "primary" vs "fallback" vs "skipped"
#   4. Parses pairs.txt for each race, samples 300+300 per race (seed=42)
#
# ⚠️  If the diagnostic in Cell 4A showed a DIFFERENT zero-padding
#    (e.g., 5 digits), update ZERO_PAD_WIDTH below before running.
#
# Expected output:
#   Resolution stats per race, pair counts, sampling summary.
# =============================================================================

import os
import re
import random
from collections import defaultdict

# ---------- Configuration ----------
# Update this if Cell 4A diagnostic shows a different zero-padding width!
ZERO_PAD_WIDTH = 4  # e.g., identity_0001.jpg → 4 digits

DATASET_ROOT = "/kaggle/input/datasets/chidroopakanaparthy/rfw-clean-400x400"
DATA_DIR = os.path.join(DATASET_ROOT, "data")
TXTS_DIR = os.path.join(DATASET_ROOT, "txts")
RACES = ["African", "Asian", "Caucasian", "Indian"]

# Verify paths exist
assert os.path.isdir(DATA_DIR), f"Data directory not found: {DATA_DIR}"
assert os.path.isdir(TXTS_DIR), f"Txts directory not found: {TXTS_DIR}"
for race in RACES:
    assert os.path.isdir(os.path.join(DATA_DIR, race)), f"Missing: {DATA_DIR}/{race}"
    assert os.path.isfile(os.path.join(TXTS_DIR, race, f"{race}_pairs.txt")), \
        f"Missing: {TXTS_DIR}/{race}/{race}_pairs.txt"

print(f"✅ Dataset located at: {DATASET_ROOT}")
print(f"   Races: {RACES}")
print(f"   Zero-pad width: {ZERO_PAD_WIDTH}")


# ---------- Resolution counters ----------
# Tracks per-race: how each image path was resolved
resolution_counts = defaultdict(lambda: {"primary": 0, "fallback": 0, "skipped": 0})


# ---------- Robust find_image_path ----------
def find_image_path(data_dir, race, identity, img_id):
    """
    Resolve an image path from a pairs file entry.
    
    Strategy:
      1. PRIMARY: Build the expected filename using ZERO_PAD_WIDTH.
         e.g., identity_0001.jpg for img_id=1 with pad=4.
      2. FALLBACK: List all files in the identity folder. For each file,
         parse the numeric portion from the filename (between last '_' and
         '.jpg'). Accept ONLY if int(parsed_numeric) == int(img_id).
         This avoids substring matches (e.g., img_id=1 matching '0012').
      3. If nothing matches, return None.
    
    Returns:
        (path, method) where method is 'primary', 'fallback', or None.
    """
    target_id = int(img_id)
    
    # --- Primary: exact expected filename ---
    filename = f"{identity}_{target_id:0{ZERO_PAD_WIDTH}d}.jpg"
    path = os.path.join(data_dir, race, identity, filename)
    if os.path.isfile(path):
        return path, "primary"
    
    # --- Fallback: list directory + parse numeric ID from each filename ---
    identity_dir = os.path.join(data_dir, race, identity)
    if not os.path.isdir(identity_dir):
        return None, None
    
    # Pattern: {anything}_{DIGITS}.{ext}
    # We extract the numeric portion after the last underscore, before the extension
    for fname in os.listdir(identity_dir):
        # Only consider image files
        name_lower = fname.lower()
        if not (name_lower.endswith('.jpg') or name_lower.endswith('.jpeg') or
                name_lower.endswith('.png')):
            continue
        
        # Extract numeric ID: everything between last '_' and the file extension
        name_no_ext = os.path.splitext(fname)[0]  # e.g., 'identity_0001'
        last_underscore = name_no_ext.rfind('_')
        if last_underscore == -1:
            continue
        
        numeric_part = name_no_ext[last_underscore + 1:]
        # Verify it's all digits
        if not numeric_part.isdigit():
            continue
        
        # STRICT match: parsed integer must equal target integer
        if int(numeric_part) == target_id:
            return os.path.join(identity_dir, fname), "fallback"
    
    return None, None


# ---------- Parse pairs files ----------
def parse_pairs_file(pairs_path, data_dir, race):
    """
    Parse an RFW pairs file.
    
    Format:
      Line 1: number_of_pairs (header, sometimes present)
      Positive pair: identity img1_id img2_id           (3 tokens)
      Negative pair: identity1 img1_id identity2 img2_id (4 tokens)
    
    Returns:
      positive_pairs: list of (img1_path, img2_path)
      negative_pairs: list of (img1_path, img2_path)
      skipped_count:  number of pairs where image path couldn't be resolved
    """
    positive_pairs = []
    negative_pairs = []
    skipped = 0
    
    with open(pairs_path, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        tokens = line.split('\t')
        # Some files use spaces instead of tabs
        if len(tokens) == 1:
            tokens = line.split()
        
        # Skip header line (single number)
        if len(tokens) == 1:
            continue
        
        if len(tokens) == 3:
            # Positive pair: same identity, two different images
            identity, img1_id, img2_id = tokens
            img1_path, method1 = find_image_path(data_dir, race, identity, img1_id)
            img2_path, method2 = find_image_path(data_dir, race, identity, img2_id)
            if img1_path and img2_path:
                positive_pairs.append((img1_path, img2_path))
                resolution_counts[race][method1] += 1
                resolution_counts[race][method2] += 1
            else:
                skipped += 1
                # Count each failed image individually
                if img1_path is None:
                    resolution_counts[race]["skipped"] += 1
                else:
                    resolution_counts[race][method1] += 1
                if img2_path is None:
                    resolution_counts[race]["skipped"] += 1
                else:
                    resolution_counts[race][method2] += 1
        
        elif len(tokens) == 4:
            # Negative pair: different identities
            identity1, img1_id, identity2, img2_id = tokens
            img1_path, method1 = find_image_path(data_dir, race, identity1, img1_id)
            img2_path, method2 = find_image_path(data_dir, race, identity2, img2_id)
            if img1_path and img2_path:
                negative_pairs.append((img1_path, img2_path))
                resolution_counts[race][method1] += 1
                resolution_counts[race][method2] += 1
            else:
                skipped += 1
                if img1_path is None:
                    resolution_counts[race]["skipped"] += 1
                else:
                    resolution_counts[race][method1] += 1
                if img2_path is None:
                    resolution_counts[race]["skipped"] += 1
                else:
                    resolution_counts[race][method2] += 1
    
    return positive_pairs, negative_pairs, skipped


# Parse all races
all_pairs = {}  # race -> {"positive": [...], "negative": [...]}

print("\n" + "=" * 60)
print("PARSING PAIRS FILES")
print("=" * 60)

for race in RACES:
    pairs_path = os.path.join(TXTS_DIR, race, f"{race}_pairs.txt")
    pos, neg, skip = parse_pairs_file(pairs_path, DATA_DIR, race)
    all_pairs[race] = {"positive": pos, "negative": neg}
    print(f"  {race:10s}: {len(pos):5d} positive, {len(neg):5d} negative "
          f"({skip} pairs skipped due to missing images)")

# ---------- Print resolution method breakdown ----------
print("\n" + "=" * 60)
print("IMAGE PATH RESOLUTION BREAKDOWN (per race, across all pairs)")
print("=" * 60)
print(f"  {'Race':10s}  {'Primary':>10s}  {'Fallback':>10s}  {'Skipped':>10s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}")
for race in RACES:
    rc = resolution_counts[race]
    print(f"  {race:10s}  {rc['primary']:10d}  {rc['fallback']:10d}  {rc['skipped']:10d}")


# ---------- Sample 300 pos + 300 neg per race ----------
random.seed(42)

SAMPLE_POS = 300
SAMPLE_NEG = 300

sampled_pairs = []  # list of dicts with race, pair_type, img1_path, img2_path, label

print("\n" + "=" * 60)
print(f"SAMPLING {SAMPLE_POS} POSITIVE + {SAMPLE_NEG} NEGATIVE PER RACE")
print("=" * 60)

for race in RACES:
    pos = all_pairs[race]["positive"]
    neg = all_pairs[race]["negative"]
    
    # Ensure we have enough pairs
    n_pos = min(SAMPLE_POS, len(pos))
    n_neg = min(SAMPLE_NEG, len(neg))
    
    if n_pos < SAMPLE_POS:
        print(f"  ⚠️  {race}: Only {len(pos)} positive pairs available "
              f"(requested {SAMPLE_POS})")
    if n_neg < SAMPLE_NEG:
        print(f"  ⚠️  {race}: Only {len(neg)} negative pairs available "
              f"(requested {SAMPLE_NEG})")
    
    sampled_pos = random.sample(pos, n_pos)
    sampled_neg = random.sample(neg, n_neg)
    
    for img1, img2 in sampled_pos:
        sampled_pairs.append({
            "race": race,
            "pair_type": "genuine",
            "img1_path": img1,
            "img2_path": img2,
            "label": 1
        })
    
    for img1, img2 in sampled_neg:
        sampled_pairs.append({
            "race": race,
            "pair_type": "impostor",
            "img1_path": img1,
            "img2_path": img2,
            "label": 0
        })
    
    print(f"  {race:10s}: sampled {n_pos} positive + {n_neg} negative")

print(f"\n✅ Total sampled pairs: {len(sampled_pairs)}")
print(f"   (Expected: {SAMPLE_POS + SAMPLE_NEG} × {len(RACES)} = "
      f"{(SAMPLE_POS + SAMPLE_NEG) * len(RACES)})")

In [ ]:
# =============================================================================
# CELL 5 — EXTRACT EMBEDDINGS & COMPUTE SIMILARITY (Step 5)
# =============================================================================
# This cell extracts embeddings for both models and computes similarities.
# 
# OPTIMIZATION: 
#   MTCNN (facenet-pytorch) is extremely slow in Python loops (takes 30+ mins).
#   Instead, we use InsightFace's C++ SCRFD detector to detect the face and
#   its 5 landmarks. This is blazing fast (takes 2 mins).
#   Since both ArcFace and AdaFace expect the exact same 112x112 affine warp 
#   based on those 5 landmarks, we crop the face once using InsightFace's 
#   alignment, and feed that aligned crop to AdaFace!
#
#   BONUS: By using the SAME face detector (SCRFD) for both models, we remove 
#   detector bias (MTCNN failing on darker skin) as a confounding factor in 
#   our fairness evaluation!
# =============================================================================

import time
import cv2
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from insightface.utils import face_align

# Cosine similarity helper
def cosine_similarity(emb1, emb2):
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

# We track skip counts if no face is detected
skip_counts = defaultdict(int)

# Track the embeddings and results
results = []
# Cache to avoid running the same image twice (many pairs share images)
# Key: img_path, Value: {"arcface": emb_array, "adaface": emb_array} or None
face_cache = {}

print("=" * 60)
print("EXTRACTING EMBEDDINGS (Optimized Shared-Detector Pipeline)")
print(f"Total pairs to process: {len(sampled_pairs)}")
print("=" * 60)

start_time = time.time()

def process_image(img_path):
    """
    Returns a dict with 'arcface' and 'adaface' embeddings, or None if detection fails.
    """
    if img_path in face_cache:
        return face_cache[img_path]
        
    # Read image using OpenCV (BGR format, which both insightface and adaface expect eventually)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        face_cache[img_path] = None
        return None
        
    # 1. Detect face & extract ArcFace embedding using InsightFace
    faces = arcface_app.get(img_bgr)
    if not faces:
        face_cache[img_path] = None
        return None
        
    # Assume largest face is the target (usually 1 face per image in RFW anyway)
    face = faces[0]
    arcface_emb = face.normed_embedding
    
    # 2. Extract AdaFace embedding using the exact same face detection
    # InsightFace gives us the 5 landmarks (face.kps). We use them to warp the face
    # to the standard 112x112 crop that AdaFace expects!
    aligned_bgr = face_align.norm_crop(img_bgr, landmark=face.kps, image_size=112)
    
    # Convert to AdaFace tensor: BGR, normalized [-1, 1], shape (1, 3, 112, 112)
    aligned_bgr = ((aligned_bgr / 255.0) - 0.5) / 0.5
    tensor = torch.tensor(aligned_bgr.transpose(2, 0, 1)).unsqueeze(0).float().cuda()
    
    with torch.no_grad():
        adaface_features, _ = adaface_model(tensor)
    
    adaface_emb = adaface_features.cpu().numpy().flatten()
    
    # Cache and return
    res = {"arcface": arcface_emb, "adaface": adaface_emb}
    face_cache[img_path] = res
    return res

# ---------------- Process Loop ----------------
for i, pair in enumerate(sampled_pairs):
    if (i + 1) % 200 == 0 or i == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        eta = (len(sampled_pairs) - i - 1) / rate if rate > 0 else 0
        print(f"  Processing Pair {i+1}/{len(sampled_pairs)} "
              f"({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")
              
    race = pair["race"]
    
    res1 = process_image(pair["img1_path"])
    res2 = process_image(pair["img2_path"])
    
    if res1 is None or res2 is None:
        skip_counts[race] += 1
        continue
        
    # Compute similarity for ArcFace
    sim_arc = cosine_similarity(res1["arcface"], res2["arcface"])
    results.append({
        "race": race,
        "pair_type": pair["pair_type"],
        "model_name": "ArcFace_buffalo_m",
        "cosine_similarity": sim_arc,
        "ground_truth_label": pair["label"]
    })
    
    # Compute similarity for AdaFace
    sim_ada = cosine_similarity(res1["adaface"], res2["adaface"])
    results.append({
        "race": race,
        "pair_type": pair["pair_type"],
        "model_name": "AdaFace_WebFace4M",
        "cosine_similarity": sim_ada,
        "ground_truth_label": pair["label"]
    })

# Cleanup
del face_cache
torch.cuda.empty_cache()

# ---------------- Save Results ----------------
df = pd.DataFrame(results)
CSV_PATH = "/kaggle/working/verification_results.csv"
df.to_csv(CSV_PATH, index=False)

total_time = time.time() - start_time

print(f"\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"Total valid pairs compared: {len(df) // 2} (out of {len(sampled_pairs)})")
print(f"Total runtime:              {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"\nResults saved to: {CSV_PATH}")

print(f"\n--- Skip counts (face detection failures by SCRFD) ---")
if not skip_counts:
    print("  (No pairs skipped — all faces detected successfully)")
else:
    for race, cnt in skip_counts.items():
        print(f"  {race}: {cnt} pairs skipped")

# Quick sanity check
for model_name in ["ArcFace_buffalo_m", "AdaFace_WebFace4M"]:
    model_df = df[df["model_name"] == model_name]
    if len(model_df) == 0:
        continue
    gen_mean = model_df[model_df["ground_truth_label"] == 1]["cosine_similarity"].mean()
    imp_mean = model_df[model_df["ground_truth_label"] == 0]["cosine_similarity"].mean()
    print(f"\n  {model_name}:")
    print(f"    Genuine mean cos-sim:  {gen_mean:.4f}")
    print(f"    Impostor mean cos-sim: {imp_mean:.4f}")
    if gen_mean <= imp_mean:
        print(f"    ⚠️  WARNING: Genuine similarity ≤ impostor — pipeline may be broken!")



In [ ]:
# =============================================================================
# CELL 6 — COMPUTE METRICS PER RACE PER MODEL (Step 6)
# =============================================================================
# This cell:
#   1. Loads results from CSV (or uses in-memory df)
#   2. For each (race, model): computes ROC curve, AUC
#   3. Picks a fixed threshold per model: the threshold that yields ~1% FPR
#      on pooled (all-races) data for that model
#   4. Computes TPR and FPR at that threshold for each race
#   5. Prints a comprehensive summary table
#
# Expected output:
#   A table with columns: Race, Model, AUC, TPR@Thresh, FPR@Thresh, N_pairs
#   AUCs should be > 0.9 for both models if the pipeline is correct.
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

# Load results (use in-memory if available, else load from CSV)
try:
    _ = df.shape  # Check if df exists from Cell 5
except NameError:
    df = pd.read_csv("/kaggle/working/verification_results.csv")

models = df["model_name"].unique()
races = ["African", "Asian", "Caucasian", "Indian"]

# ---------- 6a. Find fixed threshold per model (~1% FPR on pooled data) ----------
fixed_thresholds = {}

print("=" * 60)
print("DETERMINING FIXED THRESHOLD PER MODEL (target: 1% FPR)")
print("=" * 60)

for model_name in models:
    model_df = df[df["model_name"] == model_name]
    y_true = model_df["ground_truth_label"].values
    y_score = model_df["cosine_similarity"].values
    
    fpr_arr, tpr_arr, thresh_arr = roc_curve(y_true, y_score)
    
    # Find threshold closest to 1% FPR (0.01)
    target_fpr = 0.01
    idx = np.argmin(np.abs(fpr_arr - target_fpr))
    
    # Use the threshold that actually achieves ≤ 1% FPR
    # Walk back to find the last threshold where FPR ≤ target
    valid_idx = np.where(fpr_arr <= target_fpr)[0]
    if len(valid_idx) > 0:
        idx = valid_idx[-1]  # Highest TPR at FPR ≤ 1%
    
    threshold = thresh_arr[idx]
    fixed_thresholds[model_name] = threshold
    
    print(f"  {model_name}:")
    print(f"    Threshold for ~1% FPR: {threshold:.4f}")
    print(f"    Actual FPR at this threshold (pooled): {fpr_arr[idx]:.4f}")
    print(f"    Actual TPR at this threshold (pooled): {tpr_arr[idx]:.4f}")


# ---------- 6b. Compute per-race per-model metrics ----------
# Store ROC data for plotting in Cell 7
roc_data = {}  # (model, race) -> (fpr, tpr, auc_val)
summary_rows = []

print("\n" + "=" * 60)
print("PER-RACE PER-MODEL METRICS")
print("=" * 60)

for model_name in models:
    threshold = fixed_thresholds[model_name]
    
    for race in races:
        subset = df[(df["model_name"] == model_name) & (df["race"] == race)]
        
        if len(subset) == 0:
            continue
        
        y_true = subset["ground_truth_label"].values
        y_score = subset["cosine_similarity"].values
        
        # ROC and AUC
        fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)
        auc_val = auc(fpr_arr, tpr_arr)
        roc_data[(model_name, race)] = (fpr_arr, tpr_arr, auc_val)
        
        # TPR and FPR at fixed threshold
        y_pred = (y_score >= threshold).astype(int)
        
        # True positives, false positives, etc.
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        
        tpr_at_thresh = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr_at_thresh = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        accuracy = (tp + tn) / len(y_true)
        
        summary_rows.append({
            "Race": race,
            "Model": model_name,
            "AUC": round(auc_val, 4),
            "TPR@Thresh": round(tpr_at_thresh, 4),
            "FPR@Thresh": round(fpr_at_thresh, 4),
            "Accuracy@Thresh": round(accuracy, 4),
            "Threshold": round(threshold, 4),
            "N_pairs": len(subset),
            "N_genuine": int(np.sum(y_true == 1)),
            "N_impostor": int(np.sum(y_true == 0)),
        })

summary_df = pd.DataFrame(summary_rows)

# Pretty print
print("\n" + summary_df.to_string(index=False))

# Sanity check: flag any AUC below 0.6
low_auc = summary_df[summary_df["AUC"] < 0.6]
if len(low_auc) > 0:
    print("\n" + "⚠️" * 20)
    print("CRITICAL: Some AUCs are below 0.6 — this suggests a broken pipeline!")
    print(low_auc.to_string(index=False))
    print("STOPPING — please investigate before proceeding.")
    raise ValueError("AUC below 0.6 detected. Pipeline may be broken.")
else:
    print("\n✅ All AUCs are above 0.6 — pipeline appears healthy.")

# Save summary table for the report
summary_df.to_csv("/kaggle/working/summary_metrics.csv", index=False)
print(f"\nSummary saved to /kaggle/working/summary_metrics.csv")

In [ ]:
# =============================================================================
# CELL 7 — VISUALIZE (Step 7)
# =============================================================================
# This cell produces two plots:
#   (a) ROC curves: one subplot per model, 4 race lines overlaid
#   (b) Grouped bar chart: AUC by race, grouped by model
#
# Expected output: Two matplotlib figures displayed inline,
#   saved as roc_curves.png and auc_barplot.png
# =============================================================================

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import numpy as np

# Color palette for the 4 races — chosen for accessibility/distinctness
RACE_COLORS = {
    "African":   "#E63946",  # Red
    "Asian":     "#457B9D",  # Steel blue
    "Caucasian": "#2A9D8F",  # Teal
    "Indian":    "#E9C46A",  # Gold
}

races = ["African", "Asian", "Caucasian", "Indian"]
model_list = sorted(df["model_name"].unique())

# ==================== Plot (a): ROC Curves ====================
fig, axes = plt.subplots(1, len(model_list), figsize=(7 * len(model_list), 6),
                         squeeze=False)

for col, model_name in enumerate(model_list):
    ax = axes[0, col]
    
    for race in races:
        key = (model_name, race)
        if key not in roc_data:
            continue
        fpr_arr, tpr_arr, auc_val = roc_data[key]
        ax.plot(fpr_arr, tpr_arr,
                color=RACE_COLORS[race],
                linewidth=2,
                label=f"{race} (AUC={auc_val:.3f})")
    
    # Diagonal reference line
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Random')
    
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curves — {model_name}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
ROC_PATH = "/kaggle/working/roc_curves.png"
plt.savefig(ROC_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ ROC curves saved to {ROC_PATH}")


# ==================== Plot (b): AUC Bar Chart ====================
fig, ax = plt.subplots(figsize=(10, 6))

n_models = len(model_list)
n_races = len(races)
bar_width = 0.18
x = np.arange(n_models)

for i, race in enumerate(races):
    aucs = []
    for model_name in model_list:
        row = summary_df[(summary_df["Model"] == model_name) &
                         (summary_df["Race"] == race)]
        aucs.append(row["AUC"].values[0] if len(row) > 0 else 0)
    
    offset = (i - n_races / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, aucs, bar_width,
                  label=race, color=RACE_COLORS[race],
                  edgecolor='white', linewidth=0.8)
    
    # Add value labels on bars
    for bar, val in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8,
                fontweight='bold')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Face Verification AUC by Race and Model', fontsize=14,
             fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_list, fontsize=11)
ax.legend(title='Race', fontsize=10)
ax.set_ylim([min(summary_df['AUC'].min() - 0.05, 0.85), 1.01])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
BAR_PATH = "/kaggle/working/auc_barplot.png"
plt.savefig(BAR_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ AUC bar plot saved to {BAR_PATH}")

In [ ]:
# =============================================================================
# CELL 8 — DISPARITY SIGNIFICANCE CHECK (Step 8)
# =============================================================================
# This cell:
#   1. At the fixed threshold from Step 6, builds a
#      race × (correct/incorrect) contingency table per model
#   2. Runs chi-square test of independence
#   3. Reports chi2 statistic, p-value, and outlier races
#
# Expected output:
#   Per-model contingency tables, chi2 stats, and significance interpretation.
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

races = ["African", "Asian", "Caucasian", "Indian"]
chi2_results = []

print("=" * 60)
print("CHI-SQUARE TEST: ACCURACY DISPARITY ACROSS RACES")
print("=" * 60)

for model_name in df["model_name"].unique():
    threshold = fixed_thresholds[model_name]
    model_df = df[df["model_name"] == model_name]
    
    print(f"\n--- {model_name} (threshold={threshold:.4f}) ---")
    
    # Build contingency table: rows=races, cols=[correct, incorrect]
    contingency = []
    race_accuracies = {}
    
    for race in races:
        subset = model_df[model_df["race"] == race]
        y_true = subset["ground_truth_label"].values
        y_score = subset["cosine_similarity"].values
        y_pred = (y_score >= threshold).astype(int)
        
        correct = int(np.sum(y_pred == y_true))
        incorrect = int(np.sum(y_pred != y_true))
        accuracy = correct / (correct + incorrect) if (correct + incorrect) > 0 else 0
        
        contingency.append([correct, incorrect])
        race_accuracies[race] = accuracy
        print(f"  {race:10s}: {correct:4d} correct, {incorrect:4d} incorrect "
              f"(acc={accuracy:.4f})")
    
    contingency_arr = np.array(contingency)
    
    # Chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_arr)
    
    print(f"\n  Chi-square statistic: {chi2:.4f}")
    print(f"  Degrees of freedom:  {dof}")
    print(f"  p-value:             {p_value:.6f}")
    
    if p_value < 0.05:
        print(f"  ✅ SIGNIFICANT (p < 0.05): Accuracy differs across races.")
        
        # Identify outlier races (highest and lowest accuracy)
        best_race = max(race_accuracies, key=race_accuracies.get)
        worst_race = min(race_accuracies, key=race_accuracies.get)
        gap = race_accuracies[best_race] - race_accuracies[worst_race]
        
        print(f"  Best race:  {best_race} (acc={race_accuracies[best_race]:.4f})")
        print(f"  Worst race: {worst_race} (acc={race_accuracies[worst_race]:.4f})")
        print(f"  Gap:        {gap:.4f} ({gap*100:.1f} percentage points)")
    else:
        print(f"  ❌ NOT SIGNIFICANT (p ≥ 0.05): No strong evidence of disparity.")
        best_race = max(race_accuracies, key=race_accuracies.get)
        worst_race = min(race_accuracies, key=race_accuracies.get)
    
    chi2_results.append({
        "Model": model_name,
        "Chi2": round(chi2, 4),
        "p_value": round(p_value, 6),
        "DoF": dof,
        "Significant": "Yes" if p_value < 0.05 else "No",
        "Best_Race": best_race,
        "Best_Acc": round(race_accuracies[best_race], 4),
        "Worst_Race": worst_race,
        "Worst_Acc": round(race_accuracies[worst_race], 4),
    })

chi2_df = pd.DataFrame(chi2_results)
print("\n" + "=" * 60)
print("CHI-SQUARE SUMMARY")
print("=" * 60)
print(chi2_df.to_string(index=False))

chi2_df.to_csv("/kaggle/working/chi2_results.csv", index=False)
print(f"\n✅ Chi-square results saved to /kaggle/working/chi2_results.csv")

In [ ]:
# =============================================================================
# CELL 9 — FINAL REPORT (Step 9)
# =============================================================================
# This cell:
#   1. Generates a comprehensive markdown report
#   2. Includes executive summary, metrics table, plots, chi-square results
#   3. Records total runtime and skipped pairs
#   4. Saves as baseline_verification_report.md
#
# Expected output:
#   The report printed to stdout and saved as a .md file.
# =============================================================================

import time
import pandas as pd
from datetime import datetime

# Reload data if needed
try:
    _ = summary_df.shape
except NameError:
    summary_df = pd.read_csv("/kaggle/working/summary_metrics.csv")

try:
    _ = chi2_df.shape
except NameError:
    chi2_df = pd.read_csv("/kaggle/working/chi2_results.csv")

# Calculate overall runtime (approximate — from Cell 5 start to now)
try:
    runtime_str = f"{total_time:.1f}s ({total_time/60:.1f} min)"
except NameError:
    runtime_str = "N/A (total_time variable not available)"

# Build skip summary
skip_lines = []
try:
    for model_name in skip_counts:
        for race in ["African", "Asian", "Caucasian", "Indian"]:
            cnt = skip_counts[model_name].get(race, 0)
            if cnt > 0:
                skip_lines.append(f"| {model_name} | {race} | {cnt} |")
    if not skip_lines:
        skip_section = "No pairs were skipped — all faces detected successfully."
    else:
        skip_section = "| Model | Race | Pairs Skipped |\n|-------|------|---------------|\n"
        skip_section += "\n".join(skip_lines)
except NameError:
    skip_section = "Skip count data not available (re-run from Cell 5)."


# ---------- Build executive summary ----------
best_per_model = {}
worst_per_model = {}
for _, row in chi2_df.iterrows():
    m = row["Model"]
    best_per_model[m] = (row["Best_Race"], row["Best_Acc"])
    worst_per_model[m] = (row["Worst_Race"], row["Worst_Acc"])

sig_models = chi2_df[chi2_df["Significant"] == "Yes"]["Model"].tolist()
sig_str = "Yes" if sig_models else "No"

exec_lines = []
exec_lines.append(f"**Accuracy disparity exists:** {'Yes — statistically significant differences found.' if sig_models else 'No statistically significant differences found at p<0.05.'}")
for m in sorted(best_per_model.keys()):
    br, ba = best_per_model[m]
    wr, wa = worst_per_model[m]
    exec_lines.append(f"**{m}:** Best={br} (acc={ba:.4f}), Worst={wr} (acc={wa:.4f}), gap={ba-wa:.4f}")
exec_lines.append(f"**Statistical significance (chi-square p<0.05):** {sig_str} for model(s): {', '.join(sig_models) if sig_models else 'none'}")
exec_lines.append(f"**Sample size:** 300 genuine + 300 impostor pairs per race × 4 races = 2400 total pair-evaluations per model.")

executive_summary = "\n".join(f"{i+1}. {line}" for i, line in enumerate(exec_lines))


# ---------- Build metrics table ----------
metrics_table = summary_df.to_markdown(index=False)


# ---------- Build chi-square table ----------
chi2_table = chi2_df.to_markdown(index=False)


# ---------- Assemble report ----------
report = f"""# RFW Face-Verification Fairness Baseline Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Dataset:** RFW (Racial Faces in the Wild), 4 races  
**Models:** ArcFace (buffalo_m), AdaFace (R50/WebFace4M)  
**Sample:** 300 genuine + 300 impostor pairs per race (1200 pairs total per model)

---

## Executive Summary

{executive_summary}

---

## Verification Metrics by Race and Model

Threshold per model is set to achieve ~1% FPR on pooled (all-race) data,  
then applied uniformly across all races for fair comparison.

{metrics_table}

---

## ROC Curves

One subplot per model, with all 4 races overlaid as different colored lines.  
Disparity is visible as separation between the curves.

![ROC Curves](roc_curves.png)

---

## AUC by Race and Model

Grouped bar chart showing AUC for each race, grouped by model.

![AUC Bar Plot](auc_barplot.png)

---

## Statistical Significance (Chi-Square Test)

At the fixed threshold, we build a race × (correct/incorrect) contingency table  
per model and run a chi-square test of independence to check if accuracy  
differs significantly across races.

{chi2_table}

**Interpretation:**
- p-value < 0.05 indicates statistically significant accuracy disparity across races.
- The "Best" and "Worst" columns show which races have the highest and lowest  
  verification accuracy at the fixed threshold.

---

## Skipped Pairs (Detection Failures)

{skip_section}

---

## Runtime

Total embedding extraction time: {runtime_str}

---

## Notes

- This is a **baseline** measurement before any VLM-based mitigation.
- Thresholds are chosen to target ~1% FPR on pooled data, then applied  
  uniformly across races. This is the standard fair-comparison approach.
- The sample of 300+300 pairs per race (2400 per model total) provides  
  a solid estimate of AUC with manageable compute cost.
- If face detection fails for an image, the entire pair is skipped rather  
  than crashing. Check the "Skipped Pairs" section for counts.
"""

# Save report
REPORT_PATH = "/kaggle/working/baseline_verification_report.md"
with open(REPORT_PATH, 'w') as f:
    f.write(report)

print(report)
print("\n" + "=" * 60)
print(f"✅ Report saved to: {REPORT_PATH}")
print("=" * 60)
print("\nAll output files:")
print(f"  • {REPORT_PATH}")
print(f"  • /kaggle/working/verification_results.csv")
print(f"  • /kaggle/working/summary_metrics.csv")
print(f"  • /kaggle/working/chi2_results.csv")
print(f"  • /kaggle/working/roc_curves.png")
print(f"  • /kaggle/working/auc_barplot.png")
print(f"\n🎉 Pipeline complete!")